In [8]:
import os
from dotenv import load_dotenv
from botocore.config import Config
from langchain_aws import ChatBedrockConverse
from utils import format_messages
from deepagents.backends.utils import create_file_data
from deepagents.backends import StateBackend, FilesystemBackend, CompositeBackend
from deepagents import create_deep_agent
import json
load_dotenv()
print(os.getenv("MODEL_OPENAI"))
print(os.getenv("MODEL_ANTHROPIC"))

openai.gpt-oss-120b-1:0
arn:aws:bedrock:us-east-1:463224243948:inference-profile/us.anthropic.claude-sonnet-4-6


##### Model Configuration

In [ ]:

# boto_config = Config(
#     read_timeout=1000,        # seconds — generous headroom for long generations
#     connect_timeout=60,
#     retries={"max_attempts": 3, "mode": "adaptive"},  # rides out throttling
# )

openai_model = ChatBedrockConverse(
    model=os.getenv("MODEL_OPENAI"),
    provider="openai",
    temperature=0.0
)

anthropic_model = ChatBedrockConverse(
    model=os.getenv("MODEL_ANTHROPIC"),
    provider="anthropic",
    temperature=0.0
)

In [ ]:
transaction_id = "transaction_anzaldua_esther"

##### File Reading

In [ ]:
with open(f"ocr-markdown/{transaction_id}/F2F.md", "r") as f:
    file_content = f.read()

files = {"/documents/F2F.md": create_file_data(file_content)}


##### Classfication Agent

In [ ]:
def read_prompt_from_file(file_path):
    with open(file_path, "r") as f:
        return f.read()

In [ ]:
classification_agent = create_deep_agent(
    model = anthropic_model,
    system_prompt= read_prompt_from_file("prompts/classification_agent_system_prompt.md"),
    backend=CompositeBackend(
        default= StateBackend(),
        routes={
            "/skills/": FilesystemBackend(root_dir="/home/ubuntu/projects/e5-f2f/e5-f2f-workflow-agent/skills", virtual_mode=True)
        }
    ),
    skills = ["skills"],
)

In [ ]:
classification_agent_result = classification_agent.invoke({"messages": [{"role": "user", "content": "split the encounters"}],
                                "files": files})

##### View the Agent's response

In [ ]:
format_messages(classification_agent_result["messages"])

##### Store Classification Agent Result in a JSON file

In [ ]:

def store_json_to_file(file_name):
    json_string = json.loads(classification_agent_result['files']["/documents/F2F_classification_results.json"]['content'])
    with open(file_name, "w") as file:
        json.dump(json_string, file, indent=4)

In [ ]:
classification_output_path = f"outputs/classification/{transaction_id}.json"

store_json_to_file(classification_output_path)

##### Split encounters and store each encounter in a separate file

In [ ]:
import re
from typing import Optional
 
 
def split_document_by_encounters(document: str, encounters: list[dict]) -> list[str]:
    """
    Split a document string into encounter segments based on encounter metadata.
 
    Each encounter has:
      - page_start, page_end: inclusive page range
      - line_start, line_end: if None, split on whole pages;
                              if set, split mid-page using 1-based line numbers
                              (lines are counted within the raw document text)
      - split_anchor: optional string hint for mid-page splits (currently unused;
                      line numbers are the authoritative split point)
 
    Pages are delimited in the document by markers of the form:
        ### Page N
 
    Rules:
      - If line_start/line_end are None  → include full pages page_start..page_end.
      - If line numbers are set          → slice the raw document lines
                                           [line_start-1 : line_end]  (1-based, inclusive)
        BUT prepend the "### Page N" header of page_start if it isn't already
        the first non-blank line of the slice, so every encounter chunk clearly
        shows which page it starts on.
      - When two encounters share a page the "### Page N" header appears in BOTH
        chunks (the first encounter's slice ends before/at the header line of the
        next page; the second encounter's slice starts from its own line but we
        prepend the shared-page header so context is preserved).
    """
 
    # ── 1. Split document into numbered lines (1-based) ──────────────────────
    all_lines: list[str] = document.splitlines()  # 0-based index internally
 
    # ── 2. Build a map  page_number → line_number (1-based) of its "### Page N" marker
    page_header_line: dict[int, int] = {}          # page_num → 1-based line index
    page_header_pattern = re.compile(r'^###\s+Page\s+(\d+)\s*$')
 
    for idx, line in enumerate(all_lines):
        m = page_header_pattern.match(line.strip())
        if m:
            page_num = int(m.group(1))
            page_header_line[page_num] = idx + 1  # convert to 1-based
 
    def get_page_end_line(page_num: int) -> int:
        """Return the last 1-based line index that belongs to page_num."""
        sorted_pages = sorted(page_header_line.keys())
        page_positions = {p: page_header_line[p] for p in sorted_pages}
 
        pages_above = [p for p in sorted_pages if p > page_num]
        if pages_above:
            next_page = min(pages_above)
            return page_header_line[next_page] - 1   # line before next header
        else:
            return len(all_lines)                     # last page → end of doc
 
    def lines_to_string(line_indices_1based: range | list[int]) -> str:
        """Convert 1-based line numbers to a joined string."""
        return "\n".join(
            all_lines[i - 1] for i in line_indices_1based
            if 1 <= i <= len(all_lines)
        )
 
    # ── 3. Process each encounter ─────────────────────────────────────────────
    # Accept either a plain list or a dict with an "encounters" key
    if isinstance(encounters, dict):
        encounters = encounters.get("encounters", [])
 
    results: list[str] = []
 
    for enc in encounters:
        page_start: int        = enc["page_start"]
        page_end: int          = enc["page_end"]
        line_start: Optional[int] = enc.get("line_start")
        line_end:   Optional[int] = enc.get("line_end")
 
        # ── Case A: No line boundaries → whole pages ─────────────────────────
        if line_start is None or line_end is None:
            first_line = page_header_line.get(page_start)
            last_line  = get_page_end_line(page_end)
 
            if first_line is None:
                results.append("")
                continue
 
            chunk = lines_to_string(range(first_line, last_line + 1))
            results.append(chunk.strip())
 
        # ── Case B: Line boundaries given → slice raw lines ──────────────────
        else:
            # The raw slice
            raw_lines = list(range(line_start, line_end + 1))
            chunk_lines: list[str] = [
                all_lines[i - 1] for i in raw_lines if 1 <= i <= len(all_lines)
            ]
 
            # Ensure the ### Page <page_start> header is present at the top.
            # If the first non-blank line of the slice is NOT already that header,
            # prepend it so the encounter chunk is self-contained.
            expected_header = f"### Page {page_start}"
            first_non_blank = next(
                (ln.strip() for ln in chunk_lines if ln.strip()), ""
            )
 
            # Also: if this encounter starts mid-page (i.e. line_start is AFTER
            # the page header line for page_start), prepend the page header.
            page_header_ln = page_header_line.get(page_start)
            starts_after_header = (
                page_header_ln is not None and line_start > page_header_ln
            )
 
            if starts_after_header or first_non_blank != expected_header:
                # Only prepend if not already there
                if first_non_blank != expected_header:
                    chunk_lines = [expected_header] + chunk_lines
 
            chunk = "\n".join(chunk_lines)
            results.append(chunk.strip())
 
    return results

In [ ]:
def save_encounters_to_files(
    document: str,
    encounters: dict | list,
    root_path: str,
    transaction_id: str,
) -> list[str]:
    """
    Split a document by encounters and save each chunk as a markdown file.
 
    File path pattern:
        <root_path>/<transaction_id>/<encounter_index>.md
 
    Args:
        document:       Raw document string to split.
        encounters:     Either the full JSON dict (with "encounters" key)
                        or a plain list of encounter dicts.
        root_path:      Root directory under which files will be written.
        transaction_id: Subfolder name identifying this processing transaction.
 
    Returns:
        List of file paths that were written.
    """
    # Normalise input — accept dict or list
    if isinstance(encounters, dict):
        encounter_list = encounters.get("encounters", [])
    else:
        encounter_list = encounters
 
    # Split document into chunks
    chunks = split_document_by_encounters(document, encounter_list)
 
    # Build output folder
    output_dir = os.path.join(root_path, transaction_id)
    os.makedirs(output_dir, exist_ok=True)
 
    written: list[str] = []
 
    for enc, chunk in zip(encounter_list, chunks):
        encounter_id = enc["encounter_index"]
        file_path = os.path.join(output_dir, f"{encounter_id}.md")
 
        with open(file_path, "w", encoding="utf-8") as f:
            f.write(chunk)
 
        written.append(file_path)
        print(f"  ✓ Saved encounter {encounter_id} → {file_path}")
 
    return written

In [ ]:
ENCOUNTERS_OUTPUT_DIR = "outputs/encounters"

In [ ]:
result = save_encounters_to_files(file_content, json.loads(classification_agent_result['files']["/documents/F2F_classification_results.json"]['content'] ),
                                  ENCOUNTERS_OUTPUT_DIR, transaction_id)